# Stage 1 — Self-Supervised Continued Pretraining (Medical)
### Domain-adaptive pretraining on raw medical text → push model to the Hub

This is **notebook 1 of 3**. Each notebook is one training stage, and the
stages are chained through your **Hugging Face Hub** account:

```text
Stage 1 (this notebook): base TinyLlama  +  raw medical text  →  push  YOURNAME/med-tinyllama-stage1-pretrained
Stage 2 (SFT):           load stage1 from Hub  +  instruction data  →  push  YOURNAME/med-tinyllama-stage2-sft
Stage 3 (DPO):           load stage2 from Hub  +  preference data  →  push  YOURNAME/med-tinyllama-stage3-dpo
```

**Concept — what "self-supervised" means here.** We train a *causal* language
model on the single objective *predict the next token*. The "labels" are
just the actual next tokens in the raw text — **no human wrote them**, they
are generated automatically from the corpus. That is why this stage is
called *self-supervised*. The model learns the **language of medicine**
(terminology, drug names, scientific style) but not yet how to answer
questions — that comes in Stage 2.

## Runtime check (use a GPU)

In Colab: **Runtime → Change runtime type → T4 GPU**. QLoRA (4-bit) is what
lets a 1.1B model train on a free GPU.

In [1]:
# ============================================================
# Step 1. Install libraries
# ============================================================
!pip install -q -U datasets transformers accelerate peft bitsandbytes sentencepiece huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 106.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.9 MB/s eta 0:00:00:00:0100:01


In [2]:
# ============================================================
# Step 2. Imports
# ============================================================
import os, gc, json
from dataclasses import dataclass, asdict

import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments,
)
from peft import (
    LoraConfig, TaskType, get_peft_model,
    prepare_model_for_kbit_training, PeftModel,
)

## Hugging Face login

We push every stage's model to the Hub, and each later notebook **pulls the
previous stage's model from the Hub**. So you must be logged in.

Two ways to provide your token (create one at
https://huggingface.co/settings/tokens with *write* access):

- **Easiest in Colab:** open the key icon on the left, add a secret named
  `HF_TOKEN`, then run the cell below — it reads the secret automatically.
- **Or** just run `login()` and paste the token when prompted.

In [10]:
# ============================================================
# Hugging Face login
# ============================================================
from huggingface_hub import login, whoami

try:
    # Colab secret named HF_TOKEN (recommended).
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    # Fallback: paste the token when prompted.
    login()

HF_USERNAME = whoami()["name"]
print("Logged in as:", HF_USERNAME)

Logged in as: Mohan143


## Step 3 — Configuration

All settings live in one dataclass. The Hub repo *names* are combined with
your username (from the login cell) to form the full repo id.

**Dataset.** We use `MedRAG/textbooks`, which is open medical-textbook text
split into short passages — ideal for continued pretraining. You can swap in
any raw-text medical dataset (e.g. `ywchoi/OpenMedText`); the loader below
auto-detects the text column, so you usually only change `dataset_name`.

In [11]:
# ============================================================
# Step 3. Configuration
# ============================================================
@dataclass
class Config:
    base_model: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Raw-text medical dataset for continued pretraining.
    dataset_name: str = "MedRAG/textbooks"
    dataset_config: str = None          # set if the dataset needs a config name
    n_samples: int = 2000               # subsample for a fast Colab demo

    # Hub repo NAME for this stage's output (username added after login).
    stage1_repo_name: str = "med-tinyllama-stage1-pretrained"
    private_repo: bool = True

    # Local working dirs.
    output_dir: str = "/content/stage1_output"
    adapter_dir: str = "/content/stage1_adapter"
    merged_dir: str = "/content/stage1_merged"

    # Tokenization / packing.
    block_size: int = 512
    test_size: float = 0.1
    seed: int = 42

    # LoRA.
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    # Training.
    num_train_epochs: float = 1.0
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_steps: int = 5
    weight_decay: float = 0.01
    logging_steps: int = 5
    max_steps: int = -1                 # set to ~30 for a very quick smoke test


config = Config()
for d in (config.output_dir, config.adapter_dir, config.merged_dir):
    os.makedirs(d, exist_ok=True)

STAGE1_REPO = f"{HF_USERNAME}/{config.stage1_repo_name}"
print("This stage will push to:", STAGE1_REPO)
print(json.dumps({k: v for k, v in asdict(config).items()}, indent=2))

This stage will push to: Mohan143/med-tinyllama-stage1-pretrained
{
  "base_model": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "dataset_name": "MedRAG/textbooks",
  "dataset_config": null,
  "n_samples": 2000,
  "stage1_repo_name": "med-tinyllama-stage1-pretrained",
  "private_repo": true,
  "output_dir": "/content/stage1_output",
  "adapter_dir": "/content/stage1_adapter",
  "merged_dir": "/content/stage1_merged",
  "block_size": 512,
  "test_size": 0.1,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 1.0,
  "per_device_train_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0002,
  "warmup_steps": 5,
  "weight_decay": 0.01,
  "logging_steps": 5,
  "max_steps": -1
}


## Step 4 — Load and prepare the raw text

We download the dataset, take a small random subsample (for a quick demo),
and **auto-detect the text column** (the string column with the longest
average content). Everything downstream just reads a single `text` column.

In [12]:
# ============================================================
# Step 4. Load dataset, subsample, normalize to a single "text" column
# ============================================================
raw = load_dataset(config.dataset_name, config.dataset_config, split="train") \
        if config.dataset_config else \
        load_dataset(config.dataset_name, split="train")

# Subsample for a fast demo run.
n = min(config.n_samples, len(raw))
raw = raw.shuffle(seed=config.seed).select(range(n))


def pick_text_column(dataset):
    """Return the name of the most text-like (longest avg length) string column."""
    string_cols = [name for name, feat in dataset.features.items()
                   if getattr(feat, "dtype", None) == "string"]
    if not string_cols:
        raise ValueError(f"No string columns found in {list(dataset.features)}")
    sample = dataset.select(range(min(200, len(dataset))))
    avg_len = {col: sum(len(x or "") for x in sample[col]) / len(sample) for col in string_cols}
    best = max(avg_len, key=avg_len.get)
    print("Detected text column:", best, "| avg length:", round(avg_len[best]))
    return best


text_col = pick_text_column(raw)
raw = raw.map(lambda ex: {"text": ex[text_col]},
              remove_columns=[c for c in raw.column_names if c != "text"])
raw = raw.filter(lambda ex: ex["text"] is not None and len(ex["text"].strip()) > 50)

print(raw)
print("\nExample:\n", raw[0]["text"][:500])

README.md:   0%|          | 0.00/2.62k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

chunk/Biochemistry_Lippincott.jsonl:   0%|          | 0.00/3.19M [00:00<?, ?B/s]

chunk/Cell_Biology_Alberts.jsonl:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chunk/Anatomy_Gray.jsonl:   0%|          | 0.00/5.19M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/125847 [00:00<?, ? examples/s]

Detected text column: contents | avg length: 785


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 1999
})

Example:
 Neurology_Adams. The literature is also replete with references to a “multimodal” gait disorder in the elderly that is the result of an ostensible aging of the vestibular organ, together with impaired proprioceptive function caused by distal neuropathy in the elderly, and impaired vision. Toppling, meaning tottering and falling, occurs with brainstem and cerebellar lesions, especially in the older person following a stroke. In a related defect caused by a vestibular disorder, the patient may des


In [13]:
# Train / validation split.
split = raw.train_test_split(test_size=config.test_size, seed=config.seed)
datasets = DatasetDict({"train": split["train"], "validation": split["test"]})
print(datasets)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 1799
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 200
    })
})


## Step 5 — Tokenize and pack into fixed 512-token blocks

`block_size = 512` is the **sequence length** (not the embedding size). We
tokenize, then **pack**: concatenate all token IDs into one stream and slice
it into back-to-back 512-token blocks so almost no compute is wasted on
padding. For causal LM, **labels are a copy of `input_ids`** (the model
shifts them internally for next-token prediction).

In [14]:
# ============================================================
# Step 5. Tokenizer + tokenize + pack
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(config.base_model, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def tokenize_function(examples):
    return tokenizer(examples["text"])


def create_training_blocks(tokenized):
    all_ids, all_mask = [], []
    for ids in tokenized["input_ids"]:
        all_ids.extend(ids)
    for m in tokenized["attention_mask"]:
        all_mask.extend(m)
    usable = (len(all_ids) // config.block_size) * config.block_size
    if usable == 0:
        return {"input_ids": [], "attention_mask": [], "labels": []}
    all_ids, all_mask = all_ids[:usable], all_mask[:usable]
    id_blocks, mask_blocks = [], []
    for s in range(0, usable, config.block_size):
        id_blocks.append(all_ids[s:s + config.block_size])
        mask_blocks.append(all_mask[s:s + config.block_size])
    return {"input_ids": id_blocks, "attention_mask": mask_blocks,
            "labels": [b.copy() for b in id_blocks]}


tokenized = datasets.map(tokenize_function, batched=True,
                         remove_columns=datasets["train"].column_names,
                         desc="Tokenizing")
final = tokenized.map(create_training_blocks, batched=True,
                      desc=f"Packing into {config.block_size}-token blocks")
print(final)
if len(final["train"]) == 0:
    raise ValueError("No training blocks. Increase n_samples or lower block_size.")

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Tokenizing:   0%|          | 0/1799 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/200 [00:00<?, ? examples/s]

Packing into 512-token blocks:   0%|          | 0/1799 [00:00<?, ? examples/s]

Packing into 512-token blocks:   0%|          | 0/200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 795
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 87
    })
})


## Step 6 — Load base model in 4-bit (QLoRA) and attach a LoRA adapter

**QLoRA = 4-bit quantized base + a small trainable LoRA adapter.** The base
weights are frozen; we train only the adapter — fast, cheap, and tiny to
store.

In [15]:
# ============================================================
# Step 6. Load base model (4-bit) + LoRA adapter
# ============================================================
use_cuda = torch.cuda.is_available()
print("CUDA:", use_cuda)
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

if use_cuda:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    base_model = AutoModelForCausalLM.from_pretrained(
        config.base_model, quantization_config=bnb, device_map="auto", trust_remote_code=True)
    base_model = prepare_model_for_kbit_training(base_model)
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.base_model, torch_dtype=torch.float32, trust_remote_code=True)
base_model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=config.lora_r, lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

CUDA: True


model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## Step 7 — Train

`mlm=False` selects causal (left-to-right) language modeling rather than
BERT-style masked modeling.

> **Version note.** If `eval_strategy` errors, rename it to
> `evaluation_strategy` (older `transformers`).

In [16]:
# ============================================================
# Step 7. Data collator, training args, Trainer, train
# ============================================================
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=config.warmup_steps,
    weight_decay=config.weight_decay,
    logging_steps=config.logging_steps,
    logging_first_step=True,
    eval_strategy="steps", eval_steps=20,
    save_strategy="no",
    fp16=use_cuda, bf16=False, report_to="none", remove_unused_columns=False,
)

trainer = Trainer(model=model, args=training_args,
                  train_dataset=final["train"], eval_dataset=final["validation"],
                  data_collator=data_collator)
print("Training...")
trainer.train()
print("Done.")

Training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
20,2.256411,2.163553
40,2.175573,2.103479
60,2.119023,2.083077
80,2.151324,2.075207
100,2.104647,2.071879


Done.


## Step 8 — Merge the adapter into the base and **push to the Hub**

We fold the LoRA weights into the base (`merge_and_unload`) to get a single
standalone model, then push it to `STAGE1_REPO`. **Stage 2 will load this
exact repo as its base model.**

In [ ]:
# ============================================================
# Step 8. Save adapter, merge, push merged model to the Hub
# ============================================================
trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)
# Merge adapter -> standalone model.
del trainer

In [23]:
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()
tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)
base_fp = AutoModelForCausalLM.from_pretrained(
    config.base_model,
    dtype=torch.float16,
    device_map=None,                 # CPU merge — no bitsandbytes needed
    trust_remote_code=True,
)


# base_fp = AutoModelForCausalLM.from_pretrained(
#     config.base_model,
#     dtype = torch.float16, # if use_cuda else torch.float32,
#     device_map="auto" if use_cuda else None, trust_remote_code=True)
merged = PeftModel.from_pretrained(base_fp, config.adapter_dir).merge_and_unload()
merged.save_pretrained(config.merged_dir)
tokenizer.save_pretrained(config.merged_dir)

# Push the merged model + tokenizer to the Hub.
merged.push_to_hub(STAGE1_REPO, private=config.private_repo)
tokenizer.push_to_hub(STAGE1_REPO, private=config.private_repo)
print(f"Stage 1 merged model pushed to: https://huggingface.co/{STAGE1_REPO}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...irksuoh/model.safetensors:   1%|          | 16.0MB / 2.20GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Stage 1 merged model pushed to: https://huggingface.co/Mohan143/med-tinyllama-stage1-pretrained


## Step 9 — Quick sanity check (text continuation)

This stage is *not* a chatbot yet, so we prompt with the **start of a
sentence** and let it continue.

In [24]:
# ============================================================
# Step 9. Continuation test
# ============================================================
merged.eval()
device = merged.device
for prompt in ["Metformin is one of the most widely prescribed",
               "The mechanism of action of statins involves"]:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = merged.generate(**inputs, max_new_tokens=80, do_sample=True,
                              temperature=0.7, top_p=0.9, repetition_penalty=1.1,
                              pad_token_id=tokenizer.eos_token_id)
    print("=" * 90)
    print(tokenizer.decode(out[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Metformin is one of the most widely prescribed medications. Its use is associated with a high incidence of adverse events, including hypoglycemia and cardiovascular events such as myocardial infarction, stroke, and coronary artery disease. In addition, metformin use has been associated with increases in glycated hemoglobin (HbA1c) and increased risk
The mechanism of action of statins involves alterations in lipoprotein metabolism and cholesterol homeostasis, resulting in decreased low-density lipoprotein (LDL) cholesterol levels. LDL particles are reduced by an increase in the production of triglyceride-rich lipoproteins, which are subsequently absorbed by the intestinal tract. These lipoprote


## Done — on to Stage 2

You now have a medical-domain base model on the Hub at `STAGE1_REPO`.
Open **notebook 2 (SFT)** and set its `stage1_repo` to this same id; it will
download this model and teach it to follow instructions.